# Tasks

Event loop напрямую с корутинами даже не умеет работать.

Но как же тогда запускать наши корутины в цикле событий?
Ответ: Надо “обернуть” их в объект класса asyncio.Task. Так как в цикле событий “крутятся” именно объекты Task, а не корутины напрямую.

Event loop оперирует  именно задачами (asyncio.Task), которые в свою очередь в себе содержат корутину, код которой и выполняется конкурентно в asyncio программе. Даже когда мы с помощью asyncio.run(...) запускали, казалось бы просто корутины, функция asyncio.run(...) “под капотом” превращала их в объекты класса asyncio.Task и после запускала в event loop.

In [ ]:
import asyncio

async def main():
  print("main started")
  await asyncio.sleep(1)
  print("main stopping...")


# Внутри функции asyncio.run корутина main сначала
# будет обёрнута в объект Task,
# который и запустится в цикле событий
asyncio.run(main())

Итак, Task — это объект, который оборачивает корутину и позволяет event loop управлять её выполнением. В стандартной asyncio-программе все Task работают конкурентно в одном потоке ОС. asyncio.Task является awaitable-объектом. То есть к нему можно применять оператор await.

Класс asyncio.Task унаследован от класса asyncio.Future. В этом случае говорят, что Task является Future-объектом.
asyncio.Future - это “низкоуровневый” класс, объект которого представляет будущий результат асинхронной операции.
Его часто называют "обещанием" (promise), ведь объект Future содержит в себе состояние асинхронной операции, которая ещё не завершена, но может завершиться (или не завершиться) в будущем.

Например, если вам надо асинхронно сделать запрос по сети, то можно создать объект Futurе, в который asyncio “обещает” поместить результат запроса, когда тот выполнится. Пока он не выполнен, соответственно, результата нет.

asyncio.Future можно ожидать при помощи await. Это будет означать, что мы ожидаем, пока Future не будет готов: то есть пока в нём не появится результат.

![иерархия](\images\awaitable_hierarchy.png)

Класс asyncio.Task же несёт в себе куда больше функционала, чем тот, что реализован в родительском asyncio.Future. Если asyncio.Future - это просто “коробка”, куда будет положен результат асинхронной операции, то asyncio.Task еще и должен:

*   Предоставить механизмы запуска себя, чтобы event loop их использовал
*   Когда event loop запустит Task, та в свою очередь должна запустить “в себе” корутину.

## Планирование Task в event loop (task scheduled)

В asyncio приложениях мы НЕ можем просто запустить корутину напрямую, как это делаем с обычными функциями. Ведь между нами и корутинами находится посредник - event loop, который сам запускает указанные нами корутины (обернутые в Task). Мы же должны запустить только сам event loop.

В то же время, мы НЕ можем сказать циклу событий: “Выполни сейчас вот эту задачу”. Мы с Вами можем лишь “зарегистрировать” задачу на выполнение в event loop, который в свою очередь запустит её, когда сам посчитает нужным. А точнее, когда у event loop “будет возможность” запустить нашу задачу.

Например, в event loop уже может "крутиться" несколько задач, и когда мы “регистрируем” новую в event loop, он поставит её в очередь, наряду с другими уже существующими задачами. И когда до неё дойдет очередь, event loop запустит и нашу задачу.

Вот эта самая “регистрация” задачи в event loop на выполнение и называется “планированием к выполнению”. То есть “запланировать задачу” (schedule task) - это всё равно, что сказать циклу событий: “Возьми задачу и начни ее выполнять, как только предоставится такая возможность”.

Создание и планирование Tasks

1.  asyncio.create_task(coro, *,name=None, context=None)  - функция из высокоуровневого API asyncio (более предпочтительный из всех).

*   coro - объект корутины, которая будет обёрнута в Task и запланирована к выполнению в текущем event loop.
*   name - Имя, которое будет назначено объекту Task. Опциональный аргумент, по умолчанию None. Если не задан, то имя задаче присвоится автоматически.
*   context - можно передать переменные контекста.

Функция asyncio.create_task возвращает объект Task, созданный из переданной корутины, и сразу планирует этот Task на выполнение в текущем event loop.
Данную функцию нельзя вызывать вне запущенного event loop - вылетит ошибка. (Если заглянуть в код функции asyncio.create_task, увидим, что она получает объект event loop методом asyncio.get_running_loop(). Поэтому и нельзя ее использовать вне работающего цикла событий.)

То есть asyncio.create_task мы вызываем только из корутин

2.  asyncio.ensure_future(coro_or_future, *, loop=None) – низкоуровневый метод. 

*   coro_or_future - либо объект корутины, либо future-подобный объект. Если передан future-объект, то вернет его напрямую. Если корутина, то ещё и обернет ее в Task, и запланирует к выполнению.
*   loop - Объект цикла событий, в котором корутина будет запланирована к выполнению. Опциональный параметр, по умолчанию None. Если не передан, то корутина запланируется в текущем event loop.

Если в аргумент coro_or_future передать корутину, то asyncio.ensure_future создаст на основе этой корутины объект Task,  запланирует его в указанном event loop и вернет.

3.   loop.create_task(coro, *,name=None, context=None) - низкоуровневый метод. Вызывается от конкретного объекта event loop, следовательно корутина планируется в этом самом loop.

*   coro - объект корутины, которая будет обёрнута в Task и запланирована к выполнению
*   name - Имя, которое будет назначено объекту Task. Опциональный аргумент, по умолчанию None. Если не задан, то имя задаче присвоится автоматически.
*   context - можно передать переменные контекста. Опциональный аргумент, по умолчанию None. 

Метод loop.create_task возвращает объект Task, созданный из переданной корутины, и сразу планирует этот Task на выполнение в том event loop, от которого вызван метод.

## Пример

Пусть одна корутина у нас будет в цикле for выводить сообщение «PING», а другая «PONG». И нужно, чтобы event loop поочередно запускал эти корутины, дабы в консоли нам увидеть чередующиеся сообщения PING и PONG.

In [ ]:
async def ping_coro():
    for i in range(3):
        print(f"{i} PING")
        await asyncio.sleep(0.1)  # Засыпаем на короткий промежуток просто для того, чтобы отдать право выполняться другой корутине

async def pong_coro():
    for i in range(3):
        print(f"{i} PONG")
        await asyncio.sleep(0.1)  # Засыпаем на короткий промежуток просто для того, чтобы отдать право выполняться другой корутине

Принцип работы будет следующий: когда одна корутина вывела свое сообщение, например «PING», она засыпает с помощью await asyncio.sleep(0.1) на очень малое количество времени. Event loop увидев, что задача PING приостановилась, отдает выполнение другой готовой к выполнению задаче PONG.  И та уже в свою очередь выводит своё сообщение «PONG»

In [3]:
async def main():
    ping_task = asyncio.create_task(ping_coro())  # Создали и запланировали корутину ping
    pong_task = asyncio.create_task(pong_coro())  # Создали и запланировали корутину pong
    print("ping/pong tasks scheduled, main start sleeping...")
    await asyncio.sleep(2) #  В этот момент, когда main заснёт, запланированные задачи ping и pong получат возможность поочередно выполняться в event loop.
    # Отдаем право выполнения специально на длительное время - 2 секунды,
    # чтобы наши корутины ping и pong в это время гарантировано успели отработать и завершиться после 3х итераций цикла for.

In [4]:
if __name__ == '__main__':
    await main()

ping/pong tasks scheduled, main start sleeping...
0 PING
0 PONG
1 PING
1 PONG
2 PING
2 PONG


## Разница между asyncio.sleep() и time.sleep()

Стоит ещё осветить различия между уже известной нам корутиной asyncio.sleep(...) и ее блокирующим выполнение аналогом - функцией time.sleep(...). Когда в синхронной программе мы хотим “заснуть” на какое то время, скажем на 5 секунд, используя функцию time.sleep(5), мы можем гарантировать, что ровно через 5 секунд после запуска time.sleep(5) программа продолжит своё выполнение дальше.

In [5]:
import time

def main():
  print("Start main function")
  start_time = time.time()  # Сохраняем время начала сна
  time.sleep(5)  # Засыпаем на 5 секунд, поток выполнения блокируется
  end_time = time.time()  # Сохраняем время окончания сна
  print(f"Stop sleeping. Real sleep time seconds: {int(end_time - start_time)}")  # Засечённое время доказывает, что проспали 5 секунд

if __name__ == '__main__':
  main()

Start main function
Stop sleeping. Real sleep time seconds: 5


В случае же asyncio.sleep() мы НЕ можем сказать: “Наша корутина заснёт на 5 секунд, а по истечение 5 секунд сразу продолжит выполнение”. Мы можем лишь утверждать, что “наша корутина продолжит своё выполнение НЕ раньше, чем через 5 секунд”.

Почему это так?

Потому что корутины (конечно же, обёрнутые в Task) выполняются конкурентно в цикле событий. 

И когда наша корутина приостановит своё выполнение конструкцией await asyncio.sleep(5), цикл событий сразу передаст право выполняться другой корутине, до тех пор, пока та так же сама не отдаст право выполняться следующей в очереди.  

И если предположить, что возможность выполняться перейдет к нашей корутине более чем через 5 секунд, то фактически будет так, что наша кортуна остановилась на время, большее чем 5 секунд.

Простой случай, когда в asyncio-программе у нас всего одна корутина, которая и будет засыпать на 5 секунд.

Тут результат будет схожий с синхронным вариантом - корутина продолжит работу ровно через 5 секунд после сна, так как других корутин в цикле нет, и некому задерживать выполнение.

In [6]:
import asyncio
import time

async def main():
  print("Start sleeping")
  start_time = time.time()  # Сохраняем время начала сна
  await asyncio.sleep(5)  # Отдаем право выполняться другим, как минимум на 5 секунд
  end_time = time.time()  # Сохраняем время окончания сна

  print(f"Stop sleeping. Real sleep time seconds: {int(end_time - start_time)}")

await main()

Start sleeping
Stop sleeping. Real sleep time seconds: 5


Сэмулируем ситуацию, когда в программе находятся две корутины. Первая засыпает на 5 секунд с помощью await asyncio.sleep(5) и передает право выполняться второй корутине. 

Вторая же, запустившись, долго не отдает право выполняться, например из-за длительных расчетов. Мы же в нашем примере сэмулируем эти “длительные расчеты” использованием блокирующей функции time.sleep(). 

В этом случае, реальное время приостановки первой корутины будет куда больше 5 секунд.

In [7]:
import asyncio
import time

async def coro2():
  print("Start long counting into coro2")
  time.sleep(10)  # имитируем длительную блокирующую операцию при помощи блокирующего сна на 10 секунд
  print("Stop long counting into coro2")

async def coro1():

  # Планируем на выпонение coro2. Она будет запущена в event loop независимо от coro1, как только 

   # предоставится такая возможность (как только coro1 отпустит права выполняться)
  asyncio.create_task(coro2())

  print("Start sleeping into coro1")
  start_time = time.time()  # Сохраняем время начала сна
  await asyncio.sleep(5)  # Отдаем право выполняться другим, как минимум на 5 секунд
  end_time = time.time()  # Сохраняем время окончания сна

  # Увидим, что coro1 в реальности прервалась не на 5, а аж на 10 секунд
  print(f"Stop sleeping into coro1. Real sleep time seconds: {int(end_time - start_time)}")


if __name__ == '__main__':
  await coro1()

Start sleeping into coro1
Start long counting into coro2
Stop long counting into coro2
Stop sleeping into coro1. Real sleep time seconds: 10


time.sleep(...) — блокирующая операция, в отличие от "await asyncio.sleep(...)". 
Вызвав time.sleep(...) корутина не отдаёт право выполнения, поэтому event loop в это время не может запустить другие Task.

Хоть мы и можем запланировать Task на независимый запуск (например с помощью asyncio.create_task(...)), это не означает, что корутина будет запущена немедленно, прямо в этот же момент. 

По факту корутина НЕ запустится до тех пор, пока event loop не получит возможность перейти к выполнению этой корутины (ведь стандартная асинхронная программа работает в рамках одного потока ОС. Иными словами, корутина будучи запланированной к выполнению, перед тем как реально начать работать, должна дождаться своей очереди на выполнение в event loop.

Например, представим, что корутина main запланировала запуск корутины coro через asyncio.create_task. Но coro не начнёт выполнение, пока работает main

In [8]:
import asyncio


async def coro():
    print("Coro executed")


async def main():

    task = asyncio.create_task(coro())

    print("main working.") # Тут main всё еще не отдала право выполняться, и task ожидает своей очереди

    print("main still working")  # И тут main всё еще не отдает право выполняться

    print("main awaiting task...")

    await task  # Тут main отдает право выполняться, ожидая окончания выполнения task. Следовательно у task появится возможность запуститься

    print("main stopping...")


if __name__ == '__main__':
    await main()
    # asyncio.run(main())

main working.
main still working
main awaiting task...
Coro executed
main stopping...


Тут, хоть корутина coro и запланирована в виде задачи task на выполнение, но начнёт выполняться она только тогда, когда main вызовет await task, тем самым отпустив право выполняться. 

Вызвав await task корутина main как бы говорит циклу событий: “отпускаю право выполняться до тех пор, пока задача task не будет выполнена до конца.” То есть задача с корутиной main ожидает завершения выполнения задачи task (с корутиной coro внутри) с помощью конструкции await.

## Жизненный цикл Task

Что вообще может происходить с задачей:

*   Сначала Task создается из корутины.
*   Затем Task планируется в event loop на независимое выполнение.  Здесь Task переходит в состояние «scheduled»
*   Далее, в какой то момент (когда решит event loop) Task запустится, тем самым получив состояние «running»
*   В какой то момент, выполняясь, корутина может приостановить выполнение (отдать это право другим). Например, при помощи await ожидая асинхронную операцию ввода/вывода или другую Task. Приостановившись в этом случае Task переходит в состояние «suspended»
*   Также Task может нормально завершить свою работу. Это состояние «done». При этом уже можно получить результат данной Task.
*   Или же упасть с ошибкой. В этом случае у Task тоже будет статус «done», но при этом можно получить объект исключения, возникшего внутри корутины этой Task.
*   В то же время, возможен вариант, когда какая нибудь другая задача отменила нашу task. Тут task также будет в состоянии «done», но вызвав метод task.cancelled() , мы получим true (что свидетельствует о том, что задача была отменена)

! Важный момент: когда Task находится в статусе «done», её уже нельзя будет снова запустить.

![иерархия](\images\lifecycle.png)

Результат работы Task можно получить при помощи метода task.result(). Этот метод вернет либо значение результата работы корутины, находящейся “внутри” этой Task. Либо None, если корутина явно результат не возвращает.
Важный момент: метод result() можно вызывать только у Task, которые уже перешли в статус “done”.
Если же мы у Task методом task.result() запросим результат до того, как она перейдёт в состояние “done”, то метод task.result() сгенерирует исключение  InvalidStateError

In [ ]:
import asyncio

async def coro():
  print("--- coro started")
  await asyncio.sleep(1)
  print("--- coro stopping...")
  return 1

async def main():
  print("--- main started")
  task = asyncio.create_task(coro())  # Оборачиваем корутину в Task и планируем к выполнению
  print("--- main start sleeping")
  
  # если поставить время меньше 1 с, то выпадет исключение
  # await asyncio.sleep(2) 
  # Дожидаемся завершения выполнения task с корутиной coro внутри
  await task 
  # result = await task -- положить результат в result
  
  print("--- main stop sleeping")
  result = task.result()  # 2 секунды было достаточно, чтобы coro отработала до конца. Поэтому результат точно уже есть
  print("Result:", result)

if __name__ == '__main__':
    await main()
#   asyncio.run(main())

--- main started
--- main start sleeping
--- coro started
--- coro stopping...
--- main stop sleeping
Result: 1


Когда мы применяем await к объекту Task, то “под капотом” происходит буквально следующее:

*   Ожидание завершения Task
*   По завершении Task, вызывается метод .result() от её объекта, и возвращается значение результата

In [10]:
async def my_coro():
    print('my_coro запущена')
    await asyncio.sleep(1)
    return 'это результат'

async def main():

    task = asyncio.create_task(my_coro())
    res = await task
    print(f'Получен результат: {res}')
    
    try:
        res = await task
    except Exception as e:
        print('возникло исключение при повторном вызове await task')
    else:
        print(f'Ещё раз получен результат: {res}')

if __name__ == '__main__':
    await main()

my_coro запущена
Получен результат: это результат
Ещё раз получен результат: это результат


В корутине, обёрнутой в Task, может возникнуть необработанное исключение. Мы можем получить объект этого исключения при помощи метода task.exception(). Метод .exception() вернёт объект исключения, если внутри кортутины оно возникло и не было обработано. Если же корутина завершилась без ошибок, то метод .exception() вернёт None.

Не всегда безопасно вызывать метод task.exception()

Как мы помним из урока по жизненному циклу Task, что если она завершилась ненормальным образом (не отработав до конца): в результате необработанного исключения или в результате отмены со стороны другой задачи, то она всё равно будет в завершённом состоянии (в статусе “done”). 

Следовательно, метод exception() можно вызывать только у завершенных Task, иначе он выбросит исключение InvalidStateError.

In [ ]:
task = asyncio.create_task()

In [ ]:
if not task.done():
  await task
exception = task.exception()

try:
  exception = task.exception()
except asyncio.InvalidStateError:
  pass

In [ ]:
if not task.cancelled():
  exception = task.exception()

try:
  exception = task.exception()
except asyncio.CancelledError:
  pass

А если попытаться получить результат у задачи, в которой возникло необработанное исключение?
Если внутри Task возникло необработанное исключение, то вызов task.result() выбросит это самое исключение. 

То есть метод .result() “прокинет” исключение вверх в вызывающий его объект.

В корутине coro намеренно выбрасываем исключение, чтобы понаблюдать за происходящим в основной задаче с корутиной main:

In [17]:
import asyncio

# Создаём собственный класс исключения
class TestException(Exception):
  pass

async def coro():
  print("--- coro started")
  await asyncio.sleep(0.5)
  raise TestException("Test exception raised!!")  # Намеренно выбрасываем наше тестовое исключение

async def main():
  print("--- main started")
  task = asyncio.create_task(coro())  # Создаём Task из корутины coro и планируем на выполнение
  result = -1
  print("--- Try to await task with coro and fetch result")
  try:
      result = await task  # Пытаемся подождать завершения task и вернуть её результат
  except Exception as e:
      print("Exception occurred:", type(e),  e)
  print("Task done?", task.done())
  print("Task exception class:", type(task.exception()), ", exception content:", task.exception())
  print("Task result:", result)

if __name__ == '__main__':
   await main()
#   asyncio.run(main())

--- main started
--- Try to await task with coro and fetch result
--- coro started
Exception occurred: <class '__main__.TestException'> Test exception raised!!
Task done? True
Task exception class: <class '__main__.TestException'> , exception content: Test exception raised!!
Task result: -1


Попытка получения возникшего в task необработанного исключения от отменённой task с помощью task.exception() выбросит исключение CancelledError.

Если Task была отменена и мы попытаемся получить результат её работы, вызвав метод task.result(), то этот метод выбросит исключение CancelledError. Подобное же происходит и в случае вызова await task.

## Отмена Task

Отменять задачи мы можем при помощи метода .cancel() класса asyncio.Task. Метод task.cancel() вернет True, если запрос на отмену task был успешно принят. Иначе, вернётся False. Если Task находится в статусе “done”, то её нельзя отменить. В этом случае метод .cancel() вернёт False.

In [1]:
import asyncio

async def coro():
  print("--- coro started")
  await asyncio.sleep(1)
  print("--- coro finished")

async def main():
  task = asyncio.create_task(coro())
  print("--- start awaiting coro")
  await task  # Дожидаемся завершения Task с корутиной coro, чтобы она гарантированно перешла в статус "done"
  print("--- coro awaited")

  is_cancelled = task.cancel()

  print("Is cancelled:", is_cancelled)  # Выведет False, так как задачу в статусе "done" нельзя отменить

if __name__ == '__main__':
    await main()
#   asyncio.run(main())

--- start awaiting coro
--- coro started
--- coro finished
--- coro awaited
Is cancelled: False


Рассмотрим процесс отмены по шагам (для наглядности представим, что task1 (задача с корутиной coro1 внутри) хочет отменить task2 (задачу с корутиной coro2 внутри)):

1.  task1 вызывает метод .cancel() от объекта task2. На самом деле, при вызове .cancel() целевая задача не отменяется сразу. По сути, вызов cancel() - это отправка запроса в event loop с просьбой отменить определенную Task. В англоязычной литературе это называют “Cancellation request”. То есть, иными словами, вызывая метод task2.cancel() , мы планируем отмену задачи task2 в event loop.

In [ ]:
async def coro1(task2):
  # ...
  is_cancelled = task2.cancel()
  # ...

2.  Возвращается результат метода task2.cancel() - ответ на “cancellation request” (True или False). Возвращённое значение скажет нам о том, удалось ли запланировать отмену task2 или нет. Например, если event loop обнаружит, что task2 уже в статусе “done”, ответ будет сразу False.

3.  Предположим, что ответ True. Далее, когда в цикле событий task2 дождётся своей очереди на продолжение выполнения, то event loop вместо того, чтобы запустить новый “шаг” корутины coro2, прокинет внутрь неё исключение asyncio.CancelledError. То есть внутри task2 в корутине coro2 в том месте, где код корутины был приостановлен (на каком-то из операторов await), возникнет исключение CancelledError.

4.  Далее, если внутри корутины coro2 исключение CancelledError не было перехвачено и подавлено, то задача task2 будет помечена, как отменённая, и больше она в цикле событий выполняться не будет. То есть в таком случае task2 будет успешно отменена. В аргументы метода .cancel() опционально мы можем передать текстовое сообщение, которое потом будет отображено в выброшенном исключении CancelledError.

То есть отмена Task - это не что иное, как возникновение внутри неё необработанного исключения asyncio.CancelledError. Просто прокидывает это исключение внутрь корутины именно event loop, а не мы “вручную”. Нам лишь нужно вызвать метод .cancel() от соответствующей задачи, которую желаем отменить.

Продемонстрируем механизм успешной отмены задач. Сценарий следующий: корутина main пытается отменить созданную ей задачу task, оборачивающую корутину coro. Для наглядности внутри корутины coro перехватим исключение CancelledError. Но не будем его подавлять, а перевыбросим его снова при помощи raise, чтобы задача task успешно отменилась.

In [3]:
import asyncio

async def coro():
  print("--- coro start first step")
  await asyncio.sleep(0.1)
  print("--- coro start second step")
  try:
      await asyncio.sleep(3)
  except asyncio.CancelledError as e:
      print("cancelled error occurred into coro:", e)
      raise  # Обязательно перевыбрасываем исключение, чтобы задача успешно отменилась
  print("Coro finished correctly")

async def main():
  print("--- main started")
  task = asyncio.create_task(coro())  # Оборачиваем coro в Task и планируем к выполнению

  # Отдаём право выполняться. Специально ждем подольше, чтобы coro запустившись,
  # успела пройти первый await и остановиться на втором.
  await asyncio.sleep(1)

  print("--- main try to cancellation request...")

  # В метод .cancel() передаём сообщение, которое отобразится в выброшенном исключении CancelledError
  was_cancelled = task.cancel("MAIN CANCELLED CORO")

  print("--- main result of cancellation request:", was_cancelled)  # Выведет True, ведь запрос на отмену успешно запланируется.

  # Первая проверка task на отмену, выдаст False. Потому что, сделав запрос на отмену task,
  # мы пока не отпустили право выполняться, следовательно event loop пока не успел прокинуть исключение в coro
  print("-- main 1st check of task cancellation:", task.cancelled())

  # task с корутиной main отдаёт право выполняться другим. В это время event loop, увидев, что был запрос на отмену task,
  # прокинет CancelledError внутрь корутины coro.
  await asyncio.sleep(0.1)

  print("-- main 2nd check of task cancellation:", task.cancelled())  # Вторая проверка уже выдаст True, то есть к этому моменту task успешно отменена

if __name__ == '__main__':
    await main()
#   asyncio.run(main())

--- main started
--- coro start first step
--- coro start second step
--- main try to cancellation request...
--- main result of cancellation request: True
-- main 1st check of task cancellation: False
cancelled error occurred into coro: MAIN CANCELLED CORO
-- main 2nd check of task cancellation: True


Для большего понимания точки выброса исключения CancelledError, в этом примере сделали так, чтобы отмена произошла на втором await в корутине coro. Следовательно, оборачивать в try-except будем только второй по счёту await.

Обычно, если стоит задача отловить CancelledError лучше оборачивать в try-except весь код корутины. 

Итак, корутина main создаёт и планирует к выполнению задачу task с корутиной coro внутри. Далее, main засыпает на 1 секунду, чтобы корутина coro успела пройти первый await (поспать 0.1 секунду) и остановится на втором (заснув аж на 3 секунды). После этого, main продолжает выполнение и делает запрос на отмену. Напомню ещё раз, что в этот момент, корутина coro отдала право выполняться на строчке await asyncio.sleep(3). И пока main выполняется, coro “досыпает” свои 3 секунды.

После запроса на отмену, main не отпускает право выполнения, а проверяет задачу task, отменилась ли она. И получит отрицательный результат, ведь право выполняться еще не возвращалось к задаче task, и поэтому event loop еще не “прокинул” исключение CancelledError внутрь coro.

Далее main засыпает на короткий срок с помощью конструкции await asyncio.sleep(0.1), просто чтобы отдать право выполнения. В этот момент, цикл событий видит, что очередь выполняться снова дошла до задачи task, и исполняет запрос на её отмену: вместо того, чтобы запускать  task, цикл событий прокидывает внутрь неё исключение CancelledError.

В этот момент, внутри корутины coro в строчке await asyncio.sleep(3) возникнет исключение CancelledError, которое мы отловим и перевыбросим. Таким образом, корутина coro даже не успела полностью “доспать” свои 3 секунды. Цикл событий отменил её раньше. После этого, право выполняться возвращается к main, которая проверив task на отмену, получит уже положительный результат. Ведь task на этот момент времени уже успешно отменена.

Если отловить и подавить возникшее исключение asyncio.CancelledError внутри корутины, то задача не будет отменена. Посмотрим на это на примере. В этом примере продемонстрируем, как задача task с корутиной coro внутри, обработает и подавит исключение CancelledError, и тем самым избежит отмены.

In [4]:
import asyncio

async def coro():
  try:
      await asyncio.sleep(3)
  except asyncio.CancelledError as e:
      print("exception CancelledError was processed:", e)

  print("Coro finished correctly")

async def main():
  task = asyncio.create_task(coro())

  await asyncio.sleep(0.1)  # позволяем coro запуститься

  was_cancelled = task.cancel()

  print("--- main cancellation result:", was_cancelled) # Запрос отработает успешно. Будет True

  print("--- main 1st check task cancelled:", task.cancelled())

  await asyncio.sleep(0.1)

  # Не смотря на то, что запрос на отмену отработал успешно (was_cancelled = True), даже вторая проверка покажет, что
  # task не была отменена, так как внутри coro мы обработали исключение
  print("--- main 2nd check task cancelled:", task.cancelled())


if __name__ == '__main__':
    await main()
#   asyncio.run(main())

--- main cancellation result: True
--- main 1st check task cancelled: False
exception CancelledError was processed: 
Coro finished correctly
--- main 2nd check task cancelled: False


Итак, в корутине main создаём и планируем на выполнение task с корутиной coro внутри. После чего, корутина main засыпает на короткое время, чтобы coro начала своё выполнение. Корутина coro запустившись, уходит в сон на 3 секунды. Право выполнения возвращается к main, которая делает запрос на отмену task. Запрос на отмену запланируется успешно (was_cancelled == True). Ведь на начальном этапе “противопоказаний” к отмене нет, и event loop пока не знает, что мы подавим исключение CancelledError внутри coro, и task отменена не будет. Не отпуская право выполняться, main делает первую проверку на то, не отменена ли task. Результат отрицательный, ведь event loop еще не успел “дойти” до coro и прокинуть в неё исключение CancelledError. После чего, main снова засыпает на короткий срок, и event loop, увидев запрос на отмену task, прокидывает CancelledError в корутину coro. Тем самым пробуждая её от трёхсекундного сна раньше времени. В корутине coro возникшее исключение CancelledError подавляется (т.е. НЕ выбрасывается снова) в блоке try-except, после чего корутина нормально завершает своё выполнение. Затем main снова получает право выполняться, и проверив состояние task во второй раз, также убеждается, что та так и НЕ была отменена.

Попробуем получить результат отменённой задачи методом .result()
Пусть основная Task с корутиной main создаст две задачи: task и cancel_task. После чего main уйдёт в сон. Тем временем задача cancel_task отменит задачу task. А main, проснувшись, запросит результат задачи task с помощью task.result().

В этом примере продемонстрируем, что вызов task.result() вызовет исклчючение CancelledError.

In [5]:
import asyncio

async def cancel_coro(task):
   print("--- cancel_coro Cancelling task...")
   task.cancel()

async def coro():
   print("--- coro started")
   await asyncio.sleep(2)
   print("--- coro finished")  # До сюда выполнение не дойдет, так как на предыдущем шаге возникнет CancelledError

async def main():
   print("--- main started")
   task = asyncio.create_task(coro())
   cancel_task = asyncio.create_task(cancel_coro(task))

   print("--- main start sleeping")
   await asyncio.sleep(3)

   print("--- main try to get result of task")
   try:
       result = task.result()
       print("--- main result if task: ", result)
   except asyncio.CancelledError as e:
       print("--- main CancelledError occurred!")


if __name__ == "__main__":
    await main()
#    asyncio.run(main())

--- main started
--- main start sleeping
--- coro started
--- cancel_coro Cancelling task...
--- main try to get result of task
--- main CancelledError occurred!


Попробуем дождаться результата задачи, которая по итогу будут отменена.

Пусть основная Task с корутиной main создаст две задачи: task и cancel_task. После чего main будет ожидать завершения task с помощью конструкции await task .

Тем временем задача cancel_task отменит задачу task. 

В этом примере продемонстрируем, что когда task будет отменена, конструкция await task в корутине main вызовет исключение CancelledError.

In [6]:
import asyncio

async def cancel_coro(task):
   print("--- cancel_coro Cancelling task...")
   task.cancel()

async def coro():
   print("--- coro started")
   await asyncio.sleep(2)
   print("--- coro finished")  # До сюда выполнение не дойдет, так как на предыдущем шаге возникнет CancelledError

async def main():
   print("--- main started")
   task = asyncio.create_task(coro())
   cancel_task = asyncio.create_task(cancel_coro(task))

   print("--- main starts awaiting the task")
   try:
       result = await task
       print("--- main result if task: ", result)
   except asyncio.CancelledError as e:
       print("--- main CancelledError occurred!")

if __name__ == "__main__":
    await main()
#    asyncio.run(main())

--- main started
--- main starts awaiting the task
--- coro started
--- cancel_coro Cancelling task...
--- main CancelledError occurred!


Еще одна тонкость отмены Task: если задача task1 ожидает задачу task2 через конструкцию await task2, и в этот момент ожидания вдруг кто-то отменит task1, то исключение CancelledError будет проброшено и внутрь task2.

Рассмотрим это поведение на примере. Здесь у нас основная Task с корутиной main порождает задачи:

*   sub_task - Task, которая будет просто спать 5 секунд.
*   task - Task, которая будет ожидать sub_task.
*   cancel_task - Task, которая отменит task.

После чего, корутина main ожидает завершения выполнения задачи task.

В этом примере продемонстрируем, что при отмене задачи task, исключение CancelledError сначала возникнет в самом “нижнем” звене в цепочке await-ов, то есть в задаче sub_task. После этого, это же исключение CancelledError будет подниматься вверх по цепочке await-ов вплоть до основной Task с корутиной main.


![проброс исключения](\images\Async_cancel_chain_of_tasks_1.png)

In [7]:
import asyncio


async def coro(sub_task):

   try:
       await sub_task  # Здесь исключение CancelledError будет выброшено во вторую очередь
       print("-- coro awaited sub_coro")
       return 10

   except asyncio.CancelledError as e:
       print("--- coro CancelledError occurred:", e, id(e))
       raise  # Обязательно перевыбрасываем перехваченное исключение, так как наша задача не подавить, а вывести его на экран


async def sub_coro():

   try:
       await asyncio.sleep(5)  # Здесь впервые вылетит исключение CancelledError
       print("--- sub_coro sleep successfully")
       return 100

   except asyncio.CancelledError as e:
       print("--- sub_coro CancelledError occurred:", e, id(e))
       raise  # Обязательно перевыбрасываем перехваченное исключение, так как наша задача не подавить, а вывести его на экран


async def cancelling_task(task):
   await asyncio.sleep(1)  # Немного ожидаем перед отменой, чтобы sub_coro успела погрузиться к 5-ти секундный сон
   task.cancel("CANCEL NOW!!!")


async def main():
   sub_task = asyncio.create_task(sub_coro())  # Создаём sub_task и планируем к выполнению
   task = asyncio.create_task(coro(sub_task))  # Создаём task и планируем к выполнению
   cancel_task = asyncio.create_task(cancelling_task(task))  # Создаём cancel_task и планируем к выполнению

   try:
       await task  # Здесь исключение CancelledError будет выброшено в третью очередь

   except asyncio.CancelledError as e:
       print("--- main CancelledError occurred into main:", e, id(e))

   print("--- main stopping")


if __name__ == "__main__":
    await main()
#    asyncio.run(main())

--- sub_coro CancelledError occurred: CANCEL NOW!!! 1923038771168
--- coro CancelledError occurred: CANCEL NOW!!! 1923038771168
--- main CancelledError occurred into main: CANCEL NOW!!! 1923038771168
--- main stopping


Из вывода можно проследить, что несмотря на то, что отменялась именно задача task, а не sub_task, исключение CancelledError всё равно сначала возникло в sub_task, а после уже поднялось вверх по цепочке вызовов await: в task, а потом в main.

А что, если в нижнем звене цепочки await-ов подавить исключение CancelledError? Немного изменим наш пример: теперь в “самом нижнем звене” - в корутине sub_coro мы перехватим и подавим исключение CancelledError. И посмотрим, что произойдет с отменой задачи task. Схематично это будет выглядеть так:

![картинка](\images\Async_cancle_chain_of_tasks_2.png)

In [8]:
import asyncio


async def coro(sub_task):

   try:
       await sub_task
       print("--- coro awaited sub_coro")

   except asyncio.CancelledError as e:
       print("--- coro CancelledError occurred:", e, id(e))
       raise

   print("--- coro finished normally")
   return 10


async def sub_coro():

   try:
       await asyncio.sleep(5)
       print("--- sub_coro sleep successfully")
       return 100

   except asyncio.CancelledError as e:  # Подавляем исключение
       print("--- sub_coro CancelledError occurred:", e, id(e))


async def cancelling_task(task):
   await asyncio.sleep(1)  # Немного ожидаем перед отменой, чтобы sub_coro успела погрузиться к 5-ти секундный сон
   task.cancel("CANCEL NOW!!!")


async def main():
   sub_task = asyncio.create_task(sub_coro())  # Создаём sub_task и планируем к выполнению
   task = asyncio.create_task(coro(sub_task))  # Создаём task и планируем к выполнению
   cancel_task = asyncio.create_task(cancelling_task(task))  # Создаём cancel_task и планируем к выполнению

   try:
       await task
   except asyncio.CancelledError as e:
       print("--- main CancelledError occurred into main:", e, id(e))

   print("--- main Is task cancelled:", task.cancelled())  # Проверяем, что task действительно не была отменена

   print("--- main stopping")


if __name__ == "__main__":
    await main()
#    asyncio.run(main())

--- sub_coro CancelledError occurred: CANCEL NOW!!! 1923038773072
--- coro awaited sub_coro
--- coro finished normally
--- main Is task cancelled: False
--- main stopping


Из вывода программы следует интересный результат - задача task не была отменена. 

Казалось бы, так не должно было случиться, ведь отменялась у нас именно задача task, а исключение CancelledError мы подавили только в sub_task.

Но это тонкость работы отмены: исключение CancelledError начинает возникать в самом нижнем звене цепи вызовов await, и постепенно поднимается выше. И если какое-то из “нижних звеньев” (как в нашем примере sub_task) подавит это исключение, то нужная Task отменена НЕ будет - до неё просто “не доберётся” исключение CancelledError. 

## await корутины или await Task

Все случаи использования оператора await в нашем коде можно условно разделить на три варианта:

1.  Использование await для вызова вложенных корутин в рамках одной Task. На этих await-ах не происходит прерывания потока выполнения, то есть право выполняться НЕ передаётся другим Task.


In [9]:
import asyncio

async def sub_coro():
  print("code from sub_coro started")
  await asyncio.sleep(1)  # await номер 3 - отдаёт право выполнения
 
async def coro():
  print("code from coro started")
  await sub_coro()  # await номер 2 - НЕ отдаёт право выполнения

async def main():
  print("code from main started")
  await coro() # await номер 1 - НЕ отдаёт право выполнения

if __name__ == '__main__':
    await main()
#   asyncio.run(main())

code from main started
code from coro started
code from sub_coro started


Event loop забирает право выполняться только на "await с номером 3" - где, в корутине sub_coro вызывается асинхронная операция asyncio.sleep(1).

Таким образом, несмотря на то, что у нас на пути встретились два await, поток выполнения прервался только на третьем.

Все корутины main, coro и sub_coro выполняются в рамках одного объекта Task, в который функция asyncio.run обернула основную корутину main.

Вызов вложенных корутин - это всего лишь организация кода, которая, несмотря на добавление дополнительных await, не ведёт к “лишним” приостановкам потока выполнения. Наш пример мы могли бы переписать:

In [10]:
import asyncio

async def main():
  print("code from main started")
  print("code from coro started")
  print("code from sub_coro started")
  await asyncio.sleep(1)  # Здесь отпускаем право выполнения

if __name__ == '__main__':
    await main()
#   asyncio.run(main())

code from main started
code from coro started
code from sub_coro started


2.  Использование await для вызова объектов, реализующих в себе асинхронную операцию в рамках одной Task. В этом случае право выполняться передается другим Task. 

В этом случае, у нас вызов await ведёт к выполнению асинхронной операции. И Task, внутри которой корутина сделала такой вызов, отпускает право выполняться (поток выполнения корутины прерывается). Несколько примеров:

In [ ]:
# Отправка HTTP GET-запроса при помощи асинхронной библиотеки httpx.
response = await httpx.AsyncClient().get('https://www.google.com')
# Получение данных из Redis по ключу при помощи асинхронного redis-клиента:
data = await redis.get('key')
#Получение результата SELECT запроса с помощью асинхронного PostgreSQL БД клиента asyncpg:
rows = await conn.fetch("SELECT * FROM users")
# Уже известная нам функция для “асинхронного сна”
await asyncio.sleep(5)

Стоит сделать оговорку, ведь все вышеперечисленные await-нутые объекты являются корутинами. 

*   Чем этот вариант отличается от варианта 1 с вызовом вложенных корутин?
*   Почему этот вариант прерывает поток выполнения, а вариант 1 - нет?.

Всё дело в том, что для удобства разработчиков асинхронные библиотеки предоставляют для работы именно корутины, чтобы мы просто сделали await, и тут произошла “магия” - поток выполнения прервался до окончания асинхронной операции.

Например, в случае httpx - до получения содержимого страницы www.google.com.

В реальности же, внутри этих корутин так или иначе происходит await Future-объектов. И event loop, увидев, что был сделан await Future-объекта, прерывает выполнение текущей Task до появления результата в этом самом await-нутом Future-объекте.

Как работает asyncio.sleep() изнутри.
Вот так выглядит код корутины-функции asyncio.sleep

In [ ]:
async def sleep(delay, result=None):
  """Coroutine that completes after a given time (in seconds)."""
  if delay <= 0:
      await __sleep0()
      return result

  if math.isnan(delay):
      raise ValueError("Invalid delay: NaN (not a number)")

  loop = events.get_running_loop()
  future = loop.create_future()  # Создается объект Future

  # Обращаемся к таймеру цикла событий, и просим присвоить результат нашей футуре,
  # как только пройдет указанное количество секунд
  h = loop.call_later(delay,
                      futures._set_result_unless_cancelled,
                      future, result)
  try:
      return await future  # Ожидаем появления результата в нашей футуре
  finally:
      h.cancel()

Если простым языком, то здесь метод loop.call_later просит event loop, чтобы тот, пользуясь своими “внутренними часами”, через “delay” секунд присвоил нашему объекту Future результат. 

Результат не важно какой (по умолчанию None, но можно через аргументы asyncio.sleep задать любой), главное чтобы объект Future стал “завершенным”.

Как мы помним, объект Future - это просто "низкоуровневая" коробка для будущего результата. Пока результата нет, объект Future не считается завершенным. Как только кто-либо "положил" результат в объект Future, он сразу становится завершённым, то есть result_of_fut = await future_object сразу вернёт его результат.

Соответственно, когда наша футура станет завершенной, то задача, вызвавшая await asyncio.sleep(...) , продолжит выполнение с этого места. 

Разумеется Task не обязательно продолжит выполнение именно в тот же момент, когда наш объект Future стал завершённым. Но в этот момент Task точно будет добавлен в очередь из “готовых к выполнению” задач в цикле событий.

А запустится лишь тогда, когда event loop позволит (когда до задачи дойдет очередь).


3.  Использование await с объектом Task. В этом варианте одна Task await-тит другую. Иными словами, задача ожидает завершения другой задачи. 

    1.  Строчка await task в конце концов либо вернёт результат работы task, либо выкинет исключение, если такое возникло внутри task.
    2.  Здесь мы через await не имеем дел с асинхронной операцией напрямую, но поток выполнения всё равно прервётся на время ожидания завершения целевой Task. А может и не прервётся, если целевая Task уже в статусе “done”, и следовательно ожидать её завершения не нужно. Здесь await сразу вернёт результат или выкинет исключение, если такое возникло внутри целевой Task.
    3.  В этом варианте мы имеем дело с “выстраиванием задач в очередь”. Для примера, пусть из задачи task1 мы вызываем await task2. Этим мы сигнализируем циклу событий о том, что task1 не должна продолжать выполнение до тех пор, пока конкурентная ей задача task2 не завершится.
    4.  Сразу несколько Task могут ожидать одну и ту же задачу. Например, пусть из задачи task1 был сделан  result = await task3 , и из задачи task2 тоже был сделан  result = await task3 , где task3 - ещё одна Task. По итогу, у нас и task1 и task2 отдали своё право выполняться до тех пор, пока task3 не закончит своё выполнение. В конце концов и task1 , и task2 получат результат работы task3 в переменную result. Для лучшего понимания рассмотрим пример.

In [11]:
import asyncio

async def target_coro():
  print("--- target coro started")
  await asyncio.sleep(1)
  print("--- target coro stopping...")
  return 5


async def coro(target_task):
  print("--- coro started and start awaiting target task")
  result = await target_task
  print("--- coro awaited target_task and got result:", result)

async def main():
  print("--- main started, and scheduling tasks")
  target_task = asyncio.create_task(target_coro())
  another_task = asyncio.create_task(coro(target_task))

  print("--- main awaiting target_task")
  target_task_result = await target_task
  print("--- main awaited target_task and got result:", target_task_result)

  await another_task  # Убеждаемся, что и another_task завершила свою работу.

if __name__ == '__main__':
    await main()
#   asyncio.run(main())

--- main started, and scheduling tasks
--- main awaiting target_task
--- target coro started
--- coro started and start awaiting target task
--- target coro stopping...
--- main awaited target_task and got result: 5
--- coro awaited target_task and got result: 5


1я и 2я задачи в какой то момент делают await target_task , тем самым обе ожидают завершения 3ей задачи target_task (то есть “встают в очередь после неё”). После завершения target_task обе получают результат её работы.